# 🧪 Federated Learning on 20 Newsgroups
## Shapley-based Free-Rider Detection (SVRFL-style)

本 Notebook 实现了一个完整的联邦学习实验流水线：

1. **数据集**: 20 Newsgroups（TF-IDF 向量化）
2. **模型**: 2 层 MLP
3. **联邦学习**: FedAvg
4. **免费搭车攻击**: DFR / SDFR / AFR
5. **Shapley 值**: SVRFL 论文式的轮级蒙特卡洛近似
6. **搭车者检测**: 基于特征值的 K-means 聚类
7. **结果导出**: Excel + 可视化

---

## 0️⃣ 环境准备

In [1]:
# 克隆 GitHub 仓库
!git clone https://github.com/isaac-sun/20NEWS-FL.git
%cd 20NEWS-FL

fatal: destination path '20NEWS-FL' already exists and is not an empty directory.
/content/20NEWS-FL


In [2]:
!git pull

Already up to date.


In [3]:
# 安装依赖（Colab 已预装大部分，openpyxl 可能需要安装）
!pip install -q openpyxl tqdm

In [4]:
# 检查环境
import torch
import sklearn
print(f"PyTorch: {torch.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

PyTorch: 2.10.0+cu128
scikit-learn: 1.6.1
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Using device: cuda


---
## 1️⃣ 数据加载与预处理

使用 scikit-learn 的 20 Newsgroups 数据集，TF-IDF 向量化后转为 PyTorch 张量。

In [5]:
import numpy as np
from utils.seed import set_seed
from data.newsgroups import load_newsgroups

set_seed(42)

MAX_FEATURES = 10000
VAL_RATIO = 0.1

train_ds, val_ds, test_ds, input_dim, train_labels = load_newsgroups(
    max_features=MAX_FEATURES, val_ratio=VAL_RATIO
)

print(f"输入维度 (TF-IDF features): {input_dim}")
print(f"训练集大小: {len(train_ds)}")
print(f"验证集大小: {len(val_ds)}")
print(f"测试集大小: {len(test_ds)}")
print(f"类别数: {len(np.unique(train_labels))}")

输入维度 (TF-IDF features): 10000
训练集大小: 10183
验证集大小: 1131
测试集大小: 7532
类别数: 20


---
## 2️⃣ 模型定义

2 层 MLP: `input_dim → 256 → ReLU → Dropout(0.2) → 20`

In [6]:
from models.mlp import MLP

HIDDEN_DIM = 256
NUM_CLASSES = 20

model = MLP(input_dim, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params:,}")

MLP(
  (net): Sequential(
    (0): Linear(in_features=10000, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=20, bias=True)
  )
)

总参数量: 2,565,396


---
## 3️⃣ 数据分区（IID / Non-IID）

将训练集分配给各客户端。

In [7]:
from utils.partition import iid_partition, non_iid_partition
from torch.utils.data import Subset

NUM_CLIENTS = 10
IID = True  # 改为 False 可尝试 Non-IID

set_seed(42)
if IID:
    partition = iid_partition(train_ds, NUM_CLIENTS)
    print("IID 分区")
else:
    partition = non_iid_partition(train_labels, NUM_CLIENTS)
    print("Non-IID 分区 (label skew)")

for cid in range(NUM_CLIENTS):
    print(f"  Client {cid}: {len(partition[cid])} samples")

IID 分区
  Client 0: 1019 samples
  Client 1: 1019 samples
  Client 2: 1019 samples
  Client 3: 1018 samples
  Client 4: 1018 samples
  Client 5: 1018 samples
  Client 6: 1018 samples
  Client 7: 1018 samples
  Client 8: 1018 samples
  Client 9: 1018 samples


---
## 4️⃣ 单轮联邦学习演示

先跑一轮看看效果，理解各组件的配合。

In [8]:
import copy
from torch.utils.data import DataLoader
from config import Config
from models.mlp import MLP
from fl.client import FLClient
from fl.server import FLServer
from fl.aggregation import fedavg_aggregate
from utils.metrics import evaluate_model

# 配置
demo_cfg = Config(
    num_clients=NUM_CLIENTS,
    num_rounds=1,
    local_epochs=3,
    local_lr=0.001,
    server_lr=1.0,
    participation_ratio=0.8,
    batch_size=64,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    device=DEVICE,
)

# 创建模型和服务器
model_fn = lambda: MLP(input_dim, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
model_demo = model_fn()
server = FLServer(model_demo, val_ds, test_ds, demo_cfg)

# 创建客户端
client_datasets = {cid: Subset(train_ds, partition[cid]) for cid in range(NUM_CLIENTS)}
clients = {cid: FLClient(cid, client_datasets[cid], model_fn, demo_cfg) for cid in range(NUM_CLIENTS)}

# 训练前评估
loss_before, acc_before = server.evaluate()
print(f"训练前 — Loss: {loss_before:.4f}, Accuracy: {acc_before:.4f}")

# 选择客户端并训练
set_seed(42)
selected = server.select_clients(NUM_CLIENTS, 0.8)
print(f"选中客户端: {selected}")

global_sd = server.get_global_state_dict()
updates = {}
for cid in selected:
    updates[cid] = clients[cid].train(global_sd)
    print(f"  Client {cid} 训练完成")

# 聚合
new_sd = fedavg_aggregate(global_sd, updates, demo_cfg.server_lr)
server.update_global_model(new_sd)

# 训练后评估
loss_after, acc_after = server.evaluate()
print(f"\n训练后 — Loss: {loss_after:.4f}, Accuracy: {acc_after:.4f}")
print(f"提升: {(acc_after - acc_before) * 100:.2f}%")

训练前 — Loss: 2.9966, Accuracy: 0.0459
选中客户端: [0, 1, 2, 4, 5, 7, 8, 9]
  Client 0 训练完成
  Client 1 训练完成
  Client 2 训练完成
  Client 4 训练完成
  Client 5 训练完成
  Client 7 训练完成
  Client 8 训练完成
  Client 9 训练完成

训练后 — Loss: 2.9222, Accuracy: 0.4319
提升: 38.60%


---
## 5️⃣ Shapley 值计算演示

**SVRFL 论文定义**:

$$v(S, D_v) = F(w_g^t, D_v) - F(w_S^t, D_v)$$

其中 $w_S^t = w_g^t + \frac{\eta}{|S|} \sum_{i \in S} \delta_i^t$

使用蒙特卡洛排列采样近似 Shapley 值。

In [ ]:
from contribution.shapley import estimate_round_shapley

val_loader = DataLoader(val_ds, batch_size=64)
eval_model = model_fn()

# 对齐 updates 到 global_sd 所在设备
param_device = next(iter(global_sd.values())).device
updates_aligned = {cid: {k: v.to(param_device) for k, v in upd.items()} for cid, upd in updates.items()}

# 计算上一轮的 Shapley 值
shapley_vals = estimate_round_shapley(
    eval_model, updates_aligned, global_sd, val_loader,
    server_lr=1.0, num_mc_samples=30, device=DEVICE
)

print("单轮 Shapley 值（验证损失减少量）:")
print("-" * 40)
for cid in sorted(shapley_vals.keys()):
    sv = shapley_vals[cid]
    print(f"  Client {cid}: SV = {sv:+.6f}")
print(f"\n  总和: {sum(shapley_vals.values()):.6f}")


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

---
## 6️⃣ 免费搭车攻击演示

三种攻击策略：
- **DFR**: 发送小随机噪声
- **SDFR**: 发送缩放后的上一轮全局模型变化
- **AFR**: 发送上一轮变化 + 小噪声

In [ ]:
from attacks.dfr import dfr_attack
from attacks.sdfr import sdfr_attack
from attacks.afr import afr_attack

# 生成攻击更新
dfr_update = dfr_attack(global_sd, noise_scale=0.001)
sdfr_update = sdfr_attack(global_sd, None, scale=0.5)  # 第一轮没有 prev_sd
afr_update = afr_attack(global_sd, None, noise_scale=0.0005)

# 计算更新的范数作为比较
def update_norm(update):
    return sum(v.float().norm().item() ** 2 for v in update.values()) ** 0.5

honest_norms = [update_norm(updates[cid]) for cid in selected]
print(f"诚实客户端更新范数: {np.mean(honest_norms):.4f} ± {np.std(honest_norms):.4f}")
print(f"DFR  更新范数: {update_norm(dfr_update):.6f}")
print(f"SDFR 更新范数: {update_norm(sdfr_update):.6f}")
print(f"AFR  更新范数: {update_norm(afr_update):.6f}")
print("\n↑ 搭车者更新范数远小于诚实客户端")

---
## 7️⃣ 搭车者检测演示

**特征值**: $d_i = |sv_i| / L_{cosine,i}^2$

其中 $L_{cosine,i} = 1 - \cos(w_{local,i}, w_{global})$

使用 K-means 将特征值聚为 2 类，较大质心 > h × 较小质心 则标记为搭车者。

In [ ]:
from detection.free_rider_detection import compute_feature_values, detect_free_riders
from detection.utility_score import UtilityScoreTracker

# 对齐 updates 到 global_sd 所在设备
param_device = next(iter(global_sd.values())).device

# 模拟: 将 client 0 替换为 DFR 攻击者
mixed_updates = {cid: {k: v.to(param_device) for k, v in upd.items()} for cid, upd in updates.items()}
mixed_updates[0] = dfr_attack(global_sd, noise_scale=0.001)

# 重新计算 Shapley
mixed_sv = estimate_round_shapley(
    eval_model, mixed_updates, global_sd, val_loader,
    server_lr=1.0, num_mc_samples=30, device=DEVICE
)

# 计算特征值
feat_vals = compute_feature_values(mixed_updates, mixed_sv, global_sd)
suspected = detect_free_riders(feat_vals, threshold_h=200.0)

print("检测结果（Client 0 是搭车者）:")
print("-" * 55)
for cid in sorted(mixed_sv.keys()):
    tag = "🚨 搭车者" if cid == 0 else "✅ 诚实"
    flag = "⚠️ 被检出" if suspected.get(cid, False) else ""
    print(f"  Client {cid} [{tag}]: SV={mixed_sv[cid]:+.6f}  d={feat_vals[cid]:.2e}  {flag}")

# Utility Score
tracker = UtilityScoreTracker(alpha=0.5)
tracker.update(mixed_sv)
print(f"\nUtility 得分:")
for cid in sorted(tracker.scores.keys()):
    print(f"  Client {cid}: {tracker.scores[cid]:+.6f}")


---
## 8️⃣ 🚀 完整实验运行

运行 4 个实验（约 15-30 分钟）：
1. **Baseline** — 无攻击
2. **DFR** — 噪声搭车
3. **SDFR** — 缩放增量搭车
4. **AFR** — 高级搭车

每个实验 30 轮，10 个客户端，30% 恶意客户端。

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"  # 防止 KMeans 在某些环境崩溃

import copy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

from config import Config
from utils.seed import set_seed
from data.newsgroups import load_newsgroups
from models.mlp import MLP
from fl.client import FLClient
from fl.server import FLServer
from fl.aggregation import fedavg_aggregate
from attacks.dfr import dfr_attack
from attacks.sdfr import sdfr_attack
from attacks.afr import afr_attack
from contribution.shapley import estimate_round_shapley
from detection.free_rider_detection import compute_feature_values, detect_free_riders
from detection.utility_score import UtilityScoreTracker
from utils.partition import iid_partition, non_iid_partition
from utils.export import export_results

In [ ]:
def apply_attack(attack_type, global_sd, prev_sd, cfg):
    """生成免费搭车攻击更新"""
    if attack_type == "dfr":
        return dfr_attack(global_sd, cfg.dfr_noise_scale)
    elif attack_type == "sdfr":
        return sdfr_attack(global_sd, prev_sd, cfg.sdfr_scale)
    elif attack_type == "afr":
        return afr_attack(global_sd, prev_sd, cfg.afr_noise_scale)
    raise ValueError(f"Unknown: {attack_type}")


def run_experiment(config, train_dataset, val_dataset, test_dataset,
                   input_dim, train_labels):
    """运行一个完整的联邦学习实验"""
    set_seed(config.seed)
    print(f"\n{'='*60}")
    print(f"实验: {config.experiment_name} | 攻击: {config.attack_type}")
    print(f"{'='*60}")

    # 数据分区
    if config.iid:
        partition = iid_partition(train_dataset, config.num_clients)
    else:
        partition = non_iid_partition(train_labels, config.num_clients)

    client_datasets = {
        cid: Subset(train_dataset, indices)
        for cid, indices in partition.items()
    }

    # 模型/服务器/客户端
    model_fn = lambda: MLP(input_dim, config.hidden_dim, config.num_classes).to(config.device)
    model = model_fn()
    server = FLServer(model, val_dataset, test_dataset, config)
    clients = {cid: FLClient(cid, client_datasets[cid], model_fn, config)
               for cid in range(config.num_clients)}

    # 恶意客户端
    num_mal = int(config.num_clients * config.malicious_ratio) if config.attack_type != "none" else 0
    malicious_ids = set(range(num_mal))

    # 跟踪
    utility_tracker = UtilityScoreTracker(alpha=config.utility_alpha)
    round_details = []
    test_accs, test_losses = [], []
    participation_counts = {cid: 0 for cid in range(config.num_clients)}
    cumulative_sv = {cid: 0.0 for cid in range(config.num_clients)}
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size)
    eval_model = model_fn()

    # FL 训练轮次
    for round_t in tqdm(range(config.num_rounds), desc=config.experiment_name):
        selected = server.select_clients(config.num_clients, config.participation_ratio)
        global_sd = server.get_global_state_dict()
        prev_sd = server.prev_global_state_dict

        updates = {}
        for cid in selected:
            participation_counts[cid] += 1
            if cid in malicious_ids:
                updates[cid] = apply_attack(config.attack_type, global_sd, prev_sd, config)
            else:
                updates[cid] = clients[cid].train(global_sd)

        # Shapley 估计
        shapley_vals = estimate_round_shapley(
            eval_model, updates, global_sd, val_loader,
            server_lr=config.server_lr, num_mc_samples=config.num_mc_samples,
            device=config.device,
        )

        utility_tracker.update(shapley_vals)
        feat_vals = compute_feature_values(updates, shapley_vals, global_sd)
        suspected = detect_free_riders(feat_vals, config.detection_threshold_h)

        # 聚合
        new_sd = fedavg_aggregate(global_sd, updates, config.server_lr)
        server.update_global_model(new_sd)

        # 评估
        test_loss, test_acc = server.evaluate()
        test_accs.append(test_acc)
        test_losses.append(test_loss)

        if round_t % 5 == 0 or round_t == config.num_rounds - 1:
            print(f"  Round {round_t:>2d}: acc={test_acc:.4f}  loss={test_loss:.4f}")

        for cid in selected:
            sv = shapley_vals.get(cid, 0.0)
            cumulative_sv[cid] += sv
            pc = participation_counts[cid]
            round_details.append({
                "experiment_name": config.experiment_name,
                "attack_type": config.attack_type,
                "round": round_t,
                "client_id": cid,
                "is_malicious": cid in malicious_ids,
                "participation_count_so_far": pc,
                "round_shapley_value": sv,
                "cumulative_shapley_value": cumulative_sv[cid],
                "mean_shapley_value": cumulative_sv[cid] / pc,
                "free_rider_flag": suspected.get(cid, False),
                "utility_score": utility_tracker.scores.get(cid, 0.0),
            })

    # 汇总
    honest_ids = [c for c in range(config.num_clients) if c not in malicious_ids]
    def _avg_mean_sv(ids):
        vals = [cumulative_sv[c] / max(participation_counts[c], 1) for c in ids]
        return float(np.mean(vals)) if vals else 0.0
    def _avg_cum_sv(ids):
        return float(np.mean([cumulative_sv[c] for c in ids])) if ids else 0.0

    summary = {
        "experiment_name": config.experiment_name,
        "attack_type": config.attack_type,
        "malicious_ratio": config.malicious_ratio if num_mal > 0 else 0.0,
        "final_global_accuracy": test_accs[-1],
        "final_global_loss": test_losses[-1],
        "avg_round_shapley_honest": _avg_mean_sv(honest_ids),
        "avg_round_shapley_malicious": _avg_mean_sv(list(malicious_ids)),
        "avg_cumulative_shapley_honest": _avg_cum_sv(honest_ids),
        "avg_cumulative_shapley_malicious": _avg_cum_sv(list(malicious_ids)),
        "shapley_gap_honest_vs_malicious": _avg_mean_sv(honest_ids) - _avg_mean_sv(list(malicious_ids)),
        "attack_effective": "",
        "notes": "",
    }

    return round_details, summary, test_accs, test_losses

In [ ]:
# ====== 运行全部 4 个实验 ======

base_cfg = Config(
    num_clients=10,
    num_rounds=30,
    local_epochs=3,
    local_lr=0.001,
    server_lr=1.0,
    participation_ratio=0.8,
    batch_size=64,
    hidden_dim=256,
    num_classes=20,
    max_features=10000,
    val_ratio=0.1,
    malicious_ratio=0.3,
    num_mc_samples=30,
    seed=42,
    device=DEVICE,
    results_dir="results",
)

experiments = [
    ("baseline_no_attack", "none"),
    ("attack_dfr", "dfr"),
    ("attack_sdfr", "sdfr"),
    ("attack_afr", "afr"),
]

all_details = []
all_summaries = []
all_curves = {}

# 重新加载数据（确保一致性）
set_seed(42)
train_ds, val_ds, test_ds, input_dim, train_labels = load_newsgroups(
    max_features=base_cfg.max_features, val_ratio=base_cfg.val_ratio
)

for exp_name, attack_type in experiments:
    cfg = copy.deepcopy(base_cfg)
    cfg.experiment_name = exp_name
    cfg.attack_type = attack_type

    details, summary, accs, losses = run_experiment(
        cfg, train_ds, val_ds, test_ds, input_dim, train_labels
    )
    all_details.extend(details)
    all_summaries.append(summary)
    all_curves[exp_name] = {"acc": accs, "loss": losses}

# 标记攻击效果
baseline_acc = all_summaries[0]["final_global_accuracy"]
for s in all_summaries[1:]:
    drop = baseline_acc - s["final_global_accuracy"]
    s["attack_effective"] = drop > 0.02
    s["notes"] = f"accuracy drop vs baseline: {drop:.4f}"

print("\n" + "=" * 72)
print("所有实验完成！")
print("=" * 72)

---
## 9️⃣ 结果分析与可视化

In [ ]:
# 实验汇总表
df_summary = pd.DataFrame(all_summaries)
display_cols = [
    "experiment_name", "attack_type", "final_global_accuracy",
    "final_global_loss", "avg_round_shapley_honest",
    "avg_round_shapley_malicious", "shapley_gap_honest_vs_malicious"
]
df_summary[display_cols].style.format({
    "final_global_accuracy": "{:.4f}",
    "final_global_loss": "{:.4f}",
    "avg_round_shapley_honest": "{:.6f}",
    "avg_round_shapley_malicious": "{:.6f}",
    "shapley_gap_honest_vs_malicious": "{:.6f}",
}).background_gradient(subset=["final_global_accuracy"], cmap="RdYlGn")

In [ ]:
# 测试准确率曲线
fig, ax = plt.subplots(figsize=(10, 6))
for name, data in all_curves.items():
    ax.plot(data["acc"], label=name)
ax.set_xlabel("Round")
ax.set_ylabel("Test Accuracy")
ax.set_title("全局测试准确率")
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
# 测试损失曲线
fig, ax = plt.subplots(figsize=(10, 6))
for name, data in all_curves.items():
    ax.plot(data["loss"], label=name)
ax.set_xlabel("Round")
ax.set_ylabel("Test Loss")
ax.set_title("全局测试损失")
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
# Shapley 值对比: 诚实 vs 恶意
df = pd.DataFrame(all_details)
attack_names = ["attack_dfr", "attack_sdfr", "attack_afr"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) 轮级 Shapley 值随时间变化
for exp_name in attack_names:
    sub = df[df["experiment_name"] == exp_name]
    if sub.empty:
        continue
    honest = sub[~sub["is_malicious"]].groupby("round")["round_shapley_value"].mean()
    mal = sub[sub["is_malicious"]].groupby("round")["round_shapley_value"].mean()
    label = exp_name.replace("attack_", "").upper()
    axes[0].plot(honest.index, honest.values, label=f"{label} 诚实")
    axes[0].plot(mal.index, mal.values, linestyle="--", label=f"{label} 恶意")
axes[0].set_xlabel("Round")
axes[0].set_ylabel("Mean Round Shapley Value")
axes[0].set_title("轮级 Shapley 值: 诚实 vs 恶意")
axes[0].legend(fontsize=8)
axes[0].grid(True)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

# (b) 累积 Shapley 值柱状图
bar_labels, bar_vals, bar_colors = [], [], []
for exp_name in attack_names:
    sub = df[df["experiment_name"] == exp_name]
    if sub.empty:
        continue
    last_round = sub["round"].max()
    last = sub[sub["round"] == last_round]
    label = exp_name.replace("attack_", "").upper()
    h_cum = last[~last["is_malicious"]]["cumulative_shapley_value"].mean()
    m_cum = last[last["is_malicious"]]["cumulative_shapley_value"].mean()
    bar_labels += [f"{label}\n诚实", f"{label}\n恶意"]
    bar_vals += [h_cum, m_cum]
    bar_colors += ["steelblue", "coral"]

x_pos = np.arange(len(bar_labels))
axes[1].bar(x_pos, bar_vals, color=bar_colors, alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(bar_labels, fontsize=9)
axes[1].set_ylabel("Avg Cumulative Shapley Value")
axes[1].set_title("累积 Shapley 值: 诚实 vs 恶意")
axes[1].grid(True, axis="y")
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

fig.tight_layout()
plt.show()

In [ ]:
# 每个实验中各客户端的累积 Shapley 值
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (exp_name, _) in enumerate(experiments):
    sub = df[df["experiment_name"] == exp_name]
    if sub.empty:
        axes[idx].set_title(exp_name)
        continue
    last_round = sub["round"].max()
    last = sub[sub["round"] == last_round].sort_values("client_id")

    colors = ["coral" if row["is_malicious"] else "steelblue"
              for _, row in last.iterrows()]
    axes[idx].bar(last["client_id"], last["cumulative_shapley_value"], color=colors, alpha=0.8)
    axes[idx].set_xlabel("Client ID")
    axes[idx].set_ylabel("Cumulative SV")
    axes[idx].set_title(exp_name)
    axes[idx].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    axes[idx].grid(True, axis="y")

fig.suptitle("各客户端累积 Shapley 值（蓝=诚实, 红=恶意）", fontsize=14)
fig.tight_layout()
plt.show()

---
## 🔟 导出 Excel 报告

In [ ]:
os.makedirs("results", exist_ok=True)
filepath = export_results(all_details, all_summaries, "results")
print(f"✅ Excel 已保存: {filepath}")

# 在 Colab 中直接下载
try:
    from google.colab import files
    files.download(filepath)
    print("📥 已触发下载")
except ImportError:
    print("（非 Colab 环境，跳过自动下载）")

In [ ]:
# 保存所有图表到文件
from matplotlib.backends.backend_agg import FigureCanvasAgg

# 图1: 准确率
fig1, ax1 = plt.subplots(figsize=(10, 6))
for name, data in all_curves.items():
    ax1.plot(data["acc"], label=name)
ax1.set_xlabel("Round"); ax1.set_ylabel("Test Accuracy")
ax1.set_title("Global Test Accuracy Over Rounds")
ax1.legend(); ax1.grid(True)
fig1.savefig("results/test_accuracy.png", dpi=150, bbox_inches="tight")
plt.close(fig1)

# 图2: 损失
fig2, ax2 = plt.subplots(figsize=(10, 6))
for name, data in all_curves.items():
    ax2.plot(data["loss"], label=name)
ax2.set_xlabel("Round"); ax2.set_ylabel("Test Loss")
ax2.set_title("Global Test Loss Over Rounds")
ax2.legend(); ax2.grid(True)
fig2.savefig("results/test_loss.png", dpi=150, bbox_inches="tight")
plt.close(fig2)

print("✅ 图表已保存到 results/ 目录")

# 下载所有结果
try:
    from google.colab import files
    for f in ["results/test_accuracy.png", "results/test_loss.png"]:
        files.download(f)
except ImportError:
    pass

---
## 📊 最终实验汇总

In [ ]:
print("=" * 72)
print("实验汇总")
print("=" * 72)
for s in all_summaries:
    print(
        f"  {s['experiment_name']:<22s}  "
        f"acc={s['final_global_accuracy']:.4f}  "
        f"loss={s['final_global_loss']:.4f}  "
        f"SV_honest={s['avg_round_shapley_honest']:.6f}  "
        f"SV_malicious={s['avg_round_shapley_malicious']:.6f}"
    )
print("=" * 72)
print("\n关键发现:")
print("• DFR/SDFR 攻击者的 Shapley 值显著低于诚实客户端（可检测）")
print("• AFR 攻击者的 Shapley 值与诚实客户端更接近（更难检测 — 符合预期）")
print("• 搭车攻击稀释但不投毒 → 准确率下降较小")

---
## 🔧 自定义实验

你可以修改以下配置来运行不同的实验：

| 参数 | 说明 | 默认值 |
|------|------|--------|
| `num_clients` | 客户端数量 | 10 |
| `num_rounds` | 通信轮次 | 30 |
| `local_epochs` | 本地训练轮次 | 3 |
| `local_lr` | 本地学习率 | 0.001 |
| `participation_ratio` | 每轮参与比例 | 0.8 |
| `malicious_ratio` | 恶意客户端比例 | 0.3 |
| `num_mc_samples` | MC Shapley 采样数 | 30 |
| `iid` | IID 分区 | True |
| `detection_threshold_h` | 检测阈值 h | 200.0 |

In [ ]:
# 示例: 运行单个自定义实验
# custom_cfg = Config(
#     num_clients=20,
#     num_rounds=50,
#     local_epochs=5,
#     local_lr=0.001,
#     malicious_ratio=0.4,
#     attack_type="dfr",
#     iid=False,  # Non-IID
#     device=DEVICE,
#     experiment_name="custom_noniid_dfr",
# )
# details, summary, accs, losses = run_experiment(
#     custom_cfg, train_ds, val_ds, test_ds, input_dim, train_labels
# )
# print(f"Final accuracy: {accs[-1]:.4f}")